In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu" )
print(device)

cuda


In [ ]:
df = pd.read_csv("/content/fashion-mnist_train.csv")

In [ ]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.shape

(60000, 785)

In [ ]:
torch.manual_seed(42)

In [ ]:
X = df.iloc[: , 1:].values
y = df.iloc[: , 0].values

In [ ]:
X_train , x_test , y_train , y_test = train_test_split(X , y ,test_size=0.2 , random_state=42)

In [ ]:
X_train = X_train/255.0
x_test = x_test/255.0

In [ ]:
class DataSetDeclaration(Dataset):

  def __init__(self,n_features , labels):

    self.n_features = torch.tensor(n_features , dtype=torch.float32)
    self.labels = torch.tensor(labels , dtype=torch.long)

  def __len__(self):

    return len(self.n_features)

  def __getitem__(self, index):

    return self.n_features[index] , self.labels[index]



In [ ]:
train_dataset = DataSetDeclaration(X_train, y_train)

In [ ]:
test_dataset = DataSetDeclaration(x_test , y_test)

In [ ]:
train_data_loader = DataLoader(train_dataset, batch_size=32 , shuffle=True)
test_data_loader = DataLoader(test_dataset, batch_size=32 , shuffle=False)

In [ ]:
from torch.nn.modules.activation import Sigmoid
class neuralNetwork(nn.Module):

  def __init__(self , n_features):

    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(n_features , 128),
        nn.ReLU(),
        nn.Linear(128 , 64),
        nn.ReLU(),
        nn.Linear(64,10)
    )


  def forward(self , x):
    return self.model(x)

In [ ]:
learning_rate = 0.1
epochs = 100

In [ ]:
model = neuralNetwork(X_train.shape[1])
model = model.to(device)

In [ ]:
lossed = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.SGD(model.parameters() , lr=learning_rate)

In [ ]:
for epoch in range(epochs):

  total_loss_epoch = 0

  for batch_features , batch_labels in train_data_loader:


    # move to GPU
    batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)

    # call forward pass
    outputs = model(batch_features)

    # calc loss

    loss = lossed(outputs , batch_labels)


    # clear grads

    optimizer.zero_grad()

    # backpropagation

    loss.backward()

    # gradients update

    optimizer.step()

    total_loss_epoch = total_loss_epoch + loss.item()


  average = total_loss_epoch/len(train_data_loader)
  print(f"epoch {epoch + 1 } , average loss of epoch : {average}")

epoch 1 , average loss of epoch : 0.6352872474888961
epoch 2 , average loss of epoch : 0.4304986953884363
epoch 3 , average loss of epoch : 0.3861262078657746
epoch 4 , average loss of epoch : 0.3584607255011797
epoch 5 , average loss of epoch : 0.3376494748592377
epoch 6 , average loss of epoch : 0.32276468626906474
epoch 7 , average loss of epoch : 0.3078539018382629
epoch 8 , average loss of epoch : 0.2949818898836772
epoch 9 , average loss of epoch : 0.2854692505300045
epoch 10 , average loss of epoch : 0.27467058210571604
epoch 11 , average loss of epoch : 0.26830569267148774
epoch 12 , average loss of epoch : 0.2581421597401301
epoch 13 , average loss of epoch : 0.24940819991752505
epoch 14 , average loss of epoch : 0.24444738873218497
epoch 15 , average loss of epoch : 0.2385919222868979
epoch 16 , average loss of epoch : 0.23155899402375021
epoch 17 , average loss of epoch : 0.22562562982489665
epoch 18 , average loss of epoch : 0.2202964697740972
epoch 19 , average loss of epo

In [ ]:
model.eval()

neuralNetwork(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
total = 0
correct = 0

In [ ]:
with torch.no_grad():

  for batch_features , batch_labels in test_data_loader:

    batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)
    # forward pass
    outputs = model(batch_features)

    _, predicted = torch.max(outputs , 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

  print(f"model accuracy : { correct/total * 100 } %")


model accuracy : 11.899999999999999 %


In [ ]:
correct = 0
total = 0

with torch.no_grad():

  for batch_features , batch_labels in train_data_loader:

    batch_features , batch_labels = batch_features.to(device) , batch_labels.to(device)
    # forward pass
    outputs = model(batch_features)

    _, predicted = torch.max(outputs , 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

  print(f"model accuracy : { correct/total * 100 } %")

model accuracy : 98.24791666666667 %
